# CZSC 快速入门

CZSC（缠中说禅技术分析工具）是基于缠中说禅理论的综合性量化交易 Python 库，提供技术分析、信号生成、回测和市场分析等功能。

本 notebook 系统性地介绍 czsc 库中面向用户的核心功能。

**目录：**
1. [安装与环境](#1-安装与环境)
2. [数据准备](#2-数据准备)
3. [缠论分析（CZSC）](#3-缠论分析cZSC)
4. [K线合成（BarGenerator）](#4-K线合成BarGenerator)
5. [信号系统](#5-信号系统)
6. [事件与持仓](#6-事件与持仓)
7. [多级别联立分析（CzscTrader）](#7-多级别联立分析CzscTrader)
8. [策略回测（WeightBacktest）](#8-策略回测WeightBacktest)
9. [绩效分析与可视化](#9-绩效分析与可视化)
10. [数据源连接器](#10-数据源连接器)

## 1. 安装与环境

In [1]:
# 安装 czsc（命令行执行）
# pip install czsc
# 或使用 uv：uv add czsc

import czsc

print(f"版本: {czsc.__version__}")
print(f"日期: {czsc.__date__}")
print(f"作者: {czsc.__author__}")

版本: 0.10.12
日期: 20260308
作者: zengbin93


In [2]:
# 欢迎信息与环境变量
czsc.welcome()

欢迎使用CZSC！当前版本标识为 0.10.12@20260308

交易的本质就是投入一笔钱，在若干时间后换成另一笔钱出来，其中的凭证就是交易的品种。本质上
，任何东西都可以是交易品种，所谓股票的价值，不过是引诱你把钱投进来的诱饵。应用本ID理论的
人，绝对要首先认清楚这一点。对于你投入的钱来说，那些能让你在下一时刻变成更多的钱出来的凭
证就是有价值的。

--摘自《2007-01-30 15:09 教你炒股票26：市场风险如何回避》
CZSC环境变量：czsc_min_bi_len = 6; czsc_max_bi_num = 50; 


In [3]:
# 核心环境变量
from czsc import envs

print(f"最小笔长度（czsc_min_bi_len）: {envs.get_min_bi_len()}")
print(f"最大笔数量（czsc_max_bi_num）: {envs.get_max_bi_num()}")
print(f"是否使用 Python 版本: {envs.use_python()}")

# 设置环境变量可改变默认行为（需在 import czsc 之前设置）
# import os; os.environ['czsc_min_bi_len'] = '7'  # 使用老笔定义

最小笔长度（czsc_min_bi_len）: 6
最大笔数量（czsc_max_bi_num）: 50
是否使用 Python 版本: False


In [4]:
# 核心导出概览
from czsc import (
    # 核心数据结构
    CZSC, Freq, RawBar, NewBar, Direction, Operate, Signal, Event, Position,
    # K线工具
    format_standard_kline,
    # 交易系统
    CzscSignals, CzscTrader, WeightBacktest,
    # 分析工具
    daily_performance, top_drawdowns,
    # 策略
    CzscStrategyBase, CzscJsonStrategy,
)
# BarGenerator 在 czsc.core 中
from czsc.core import BarGenerator

print("核心类导入成功！")

核心类导入成功！


## 2. 数据准备

czsc 使用 `RawBar` 作为标准 K 线数据格式，包含 symbol、dt、open、close、high、low、vol、amount 等字段。

In [5]:
from czsc import Freq, RawBar, format_standard_kline
from czsc.mock import generate_symbol_kines

# 使用 mock 模块生成模拟 K 线数据（支持：1分钟、5分钟、15分钟、30分钟、日线）
df = generate_symbol_kines('000001', '30分钟', '20240101', '20240301')
print(f"数据形状: {df.shape}")
print(f"数据列: {df.columns.tolist()}")
df.head(3)

数据形状: (610, 8)
数据列: ['dt', 'symbol', 'open', 'close', 'high', 'low', 'vol', 'amount']


,dt,symbol,open,close,high,low,vol,amount
0,2024-01-01 09:30:00,000001,100.00,99.26,100.16,98.84,96249,9583142.03
1,2024-01-01 10:00:00,000001,99.26,99.22,99.30,99.12,278979,27682094.86
2,2024-01-01 10:30:00,000001,99.22,98.89,99.41,98.57,141302,13992012.16


In [6]:
# 将 DataFrame 转换为 RawBar 列表
bars = format_standard_kline(df, freq=Freq.F30)
print(f"RawBar 数量: {len(bars)}")
print(f"RawBar 类型: {type(bars[0])}")

# 查看 RawBar 属性
bar = bars[0]
print(f"\n符号: {bar.symbol}")
print(f"时间: {bar.dt}")
print(f"频率: {bar.freq}")
print(f"开: {bar.open}, 高: {bar.high}, 低: {bar.low}, 收: {bar.close}")
print(f"成交量: {bar.vol}, 成交额: {bar.amount}")

RawBar 数量: 610
RawBar 类型: <class 'builtins.RawBar'>

符号: 000001
时间: 2024-01-01 09:30:00
频率: 30分钟
开: 100.0, 高: 100.16, 低: 98.84, 收: 99.26
成交量: 96249.0, 成交额: 9583142.03


In [ ]:
# Freq 枚举 - 支持的 K 线周期
print("支持的 K 线周期（Freq 枚举）:")
for f in [Freq.F1, Freq.F5, Freq.F15, Freq.F30, Freq.F60, Freq.D, Freq.W, Freq.M]:
    print(f"  {f}")

## 3. 缠论分析（CZSC）

`CZSC` 是缠论分析的核心类，自动识别分型（FX）、笔（BI）、线段等结构。

In [7]:
from czsc import CZSC, format_standard_kline, Freq
from czsc.mock import generate_symbol_kines

# 生成数据并创建 CZSC 分析对象
df = generate_symbol_kines('000001', '30分钟', '20230101', '20240301')
bars = format_standard_kline(df, freq=Freq.F30)
c = CZSC(bars)

print(f"品种: {c.symbol}")
print(f"周期: {c.freq}")
print(f"K线数量: {len(c.bars_raw)}")
print(f"分型数量: {len(c.fx_list)}")
print(f"笔数量: {len(c.bi_list)}")
print(f"已完成笔: {len(c.finished_bis)}")

品种: 000001
周期: 30分钟
K线数量: 860
分型数量: 240
笔数量: 50
已完成笔: 50


In [8]:
# 查看分型（FX）信息
print("=== 前 5 个分型 ===")
for i, fx in enumerate(c.fx_list[:5]):
    print(f"  分型{i+1}: 时间={fx.dt}, 类型={fx.mark}, 价格={fx.fx:.2f}")

=== 前 5 个分型 ===
  分型1: 时间=2023-12-07 13:00:00, 类型=底分型, 价格=91.92
  分型2: 时间=2023-12-07 14:30:00, 类型=顶分型, 价格=93.69
  分型3: 时间=2023-12-09 09:30:00, 类型=底分型, 价格=89.76
  分型4: 时间=2023-12-09 10:00:00, 类型=顶分型, 价格=90.60
  分型5: 时间=2023-12-09 11:30:00, 类型=底分型, 价格=89.47


In [9]:
# 查看笔（BI）信息
print("=== 前 5 笔 ===")
for i, bi in enumerate(c.bi_list[:5]):
    print(f"  笔{i+1}: 方向={bi.direction}, "
          f"时间={bi.sdt.strftime('%Y-%m-%d %H:%M')} -> {bi.edt.strftime('%Y-%m-%d %H:%M')}, "
          f"价格={bi.low:.2f} ~ {bi.high:.2f}, "
          f"长度={bi.length}")

=== 前 5 笔 ===
  笔1: 方向=向下, 时间=2023-12-07 10:00 -> 2023-12-11 11:30, 价格=84.00 ~ 94.52, 长度=29
  笔2: 方向=向上, 时间=2023-12-11 11:30 -> 2023-12-12 11:30, 价格=84.00 ~ 87.05, 长度=7
  笔3: 方向=向下, 时间=2023-12-12 11:30 -> 2023-12-19 11:30, 价格=78.79 ~ 87.05, 长度=41
  笔4: 方向=向上, 时间=2023-12-19 11:30 -> 2023-12-20 11:30, 价格=78.79 ~ 81.48, 长度=9
  笔5: 方向=向下, 时间=2023-12-20 11:30 -> 2023-12-22 11:00, 价格=78.78 ~ 81.48, 长度=17


In [10]:
# 增量更新 - 逐根 K 线更新分析结果
# CZSC 支持增量更新，适合实时分析场景
new_bar = bars[-1]  # 取最后一根 K 线演示
c.update(new_bar)   # 增量更新
print(f"更新后笔数量: {len(c.bi_list)}")

更新后笔数量: 50


In [11]:
# 查看未完成笔（ubi）
print(f"未完成笔方向: {c.ubi['direction']}")
print(f"未完成笔最高价: {c.ubi['high']}")
print(f"未完成笔最低价: {c.ubi['low']}")

未完成笔方向: 向下
未完成笔最高价: 128.08
未完成笔最低价: 122.17


In [12]:
# 获取 K 线数据 DataFrame
df_bars = c.bars_raw_df
print(f"bars_raw_df 形状: {df_bars.shape}")
df_bars.head(3)

bars_raw_df 形状: (860, 10)


,symbol,dt,freq,id,open,close,high,low,vol,amount
0,000001,2023-12-07 09:30:00,30分钟,3400,93.45,93.69,93.79,93.39,106529.0,9969067.12
1,000001,2023-12-07 10:00:00,30分钟,3401,93.69,94.04,94.52,93.46,137322.0,12898348.86
2,000001,2023-12-07 10:30:00,30分钟,3402,94.04,93.47,94.29,93.25,196421.0,18416249.90


## 4. K线合成（BarGenerator）

`BarGenerator` 用于将低周期 K 线合成为高周期 K 线，支持多周期同时合成。

In [13]:
from czsc.core import BarGenerator
from czsc import Freq, format_standard_kline
from czsc.mock import generate_symbol_kines

# 生成 30 分钟 K 线
df = generate_symbol_kines('000001', '30分钟', '20230101', '20240301')
bars = format_standard_kline(df, freq=Freq.F30)

# 创建 BarGenerator，从 30 分钟合成日线
bg = BarGenerator(base_freq='30分钟', freqs=['30分钟', '日线'], max_count=5000)

# 逐根 K 线输入
for bar in bars:
    bg.update(bar)

print(f"品种: {bg.symbol}")
print(f"基础周期: {bg.base_freq}")
print(f"已合成周期: {list(bg.bars.keys())}")
print(f"30分钟K线数量: {len(bg.bars['30分钟'])}")
print(f"日线K线数量: {len(bg.bars['日线'])}")

品种: 000001
基础周期: 30分钟
已合成周期: ['30分钟', '日线']
30分钟K线数量: 4260
日线K线数量: 426


In [14]:
# 查看合成后的日线数据
daily_bars = bg.bars['日线']
print(f"日线第一根: {daily_bars[0]}")
print(f"日线最后一根: {daily_bars[-1]}")

日线第一根: RawBar(symbol=000001, dt=2023-01-01 00:00:00, freq=D, id=0, open=100, close=96.84, high=100.16, low=95.44, vol=2127216, amount=208060067.99)
日线最后一根: RawBar(symbol=000001, dt=2024-03-01 00:00:00, freq=D, id=425, open=126.42, close=123.28, high=128.08, low=122.17, vol=1646272, amount=207063985.61999997)


## 5. 信号系统

信号（Signal）是 czsc 量化体系的基础元素，用于描述市场状态。信号函数按类别组织，通过 `signals_config` 配置使用。

In [15]:
from czsc import Signal

# Signal 是信号的基本描述单位
# 信号 key 格式：K1_K2_K3，由三个维度组成
sig = Signal(k1='30分钟', k2='倒0笔', k3='方向', v1='向上')
print(f"信号: {sig}")
print(f"信号key: {sig.key}")
print(f"信号值: {sig.value}")

信号: Signal('30分钟_倒0笔_方向_向上_任意_任意_0')
信号key: 30分钟_倒0笔_方向
信号值: 向上_任意_任意_0


In [16]:
# 信号配置格式（signals_config）
# signals_config 是信号函数的参数配置列表
# 每个配置项包含：name（函数名）、freq（K线周期）和其他参数

signals_config = [
    # MA 信号：5日均线方向
    {'name': 'czsc.signals.tas_ma_base_V221101', 'freq': '日线', 'di': 1,
     'ma_type': 'SMA', 'timeperiod': 5},
    # 双均线信号：5日和20日均线关系
    {'name': 'czsc.signals.tas_double_ma_V221203', 'freq': '日线', 'di': 1,
     'ma_seq': (5, 20), 'th': 100},
]

print("信号配置示例:")
for conf in signals_config:
    print(f"  {conf['name']} @ {conf['freq']}")

信号配置示例:
  czsc.signals.tas_ma_base_V221101 @ 日线
  czsc.signals.tas_double_ma_V221203 @ 日线


In [17]:
# 使用 CzscSignals 获取多级别信号
from czsc.core import BarGenerator
from czsc import CzscSignals, format_standard_kline, Freq
from czsc.mock import generate_symbol_kines

# 准备多周期数据
df = generate_symbol_kines('000001', '30分钟', '20230101', '20240301')
bars = format_standard_kline(df, freq=Freq.F30)

bg = BarGenerator(base_freq='30分钟', freqs=['30分钟', '日线'], max_count=5000)
for bar in bars:
    bg.update(bar)

# 创建信号计算器
cs = CzscSignals(bg)
print(f"信号计算器: {cs}")
print(f"当前信号字典 keys: {list(cs.s.keys())}")
print(f"信号数量: {len(cs.s)}")

信号计算器: <CzscSignals for 000001>
当前信号字典 keys: ['symbol', 'dt', 'freq', 'id', 'open', 'close', 'high', 'low', 'vol', 'amount', 'solid', 'upper', 'lower']
信号数量: 13


In [18]:
# 信号解析工具
from czsc import SignalsParser, get_signals_config

# SignalsParser 用于解析信号字符串
parser = SignalsParser()
print(f"信号解析器: {parser}")

信号解析器: <czsc.traders.sig_parse.SignalsParser object at 0x0000021355075E80>


## 6. 事件与持仓

信号通过逻辑组合形成事件（Event），事件驱动持仓（Position）的开关操作。

**体系结构：** Signal → Event → Position

In [19]:
from czsc import Event, Position, Operate, Signal

# Operate 定义了四种操作
print("操作类型:")
print(f"  LO (开多): {Operate.LO}")
print(f"  LE (平多): {Operate.LE}")
print(f"  SE (平空): {Operate.SE}")
print(f"  SO (开空): {Operate.SO}")

操作类型:
  LO (开多): LO
  LE (平多): LE
  SE (平空): SE
  SO (开空): SO


In [20]:
# Direction 定义方向
from czsc import Direction

print(f"向上: {Direction.Up}")
print(f"向下: {Direction.Down}")

向上: 向上
向下: 向下


In [21]:
# Position 定义完整的持仓策略
# 一个 Position 包含：
#   - 开多事件（open_event）: signals_all 全部满足时触发
#   - 平多事件（exit_event）: signals_all 全部满足时触发
#   - 开空事件（open_event_short）
#   - 平空事件（exit_event_short）
#
# 每个事件可配置：
#   - signals_all: 所有信号必须满足（AND 逻辑）
#   - signals_any: 任一信号满足即可（OR 逻辑）
#   - signals_not: 所有信号不能被满足（NOT 逻辑）

# Position 通常通过 JSON 配置文件定义，示例：
pos_config = {
    "name": "多头策略_A",
    "open_event": {
        "operate": "LO",
        "signals_all": ["30分钟_倒0笔_方向_向上"],
    },
    "exit_event": {
        "operate": "LE",
        "signals_all": ["30分钟_倒0笔_方向_向下"],
    },
}
print("持仓策略配置示例:")
import json
print(json.dumps(pos_config, indent=2, ensure_ascii=False))

持仓策略配置示例:
{
  "name": "多头策略_A",
  "open_event": {
    "operate": "LO",
    "signals_all": [
      "30分钟_倒0笔_方向_向上"
    ]
  },
  "exit_event": {
    "operate": "LE",
    "signals_all": [
      "30分钟_倒0笔_方向_向下"
    ]
  }
}


## 7. 多级别联立分析（CzscTrader）

`CzscTrader` 是多级别联立交易决策的核心类，继承自 `CzscSignals`，支持多策略独立执行。

可同时分析不同时间周期（如 30分钟、日线）进行全面的市场决策。

In [22]:
from czsc.core import BarGenerator
from czsc import CzscTrader, format_standard_kline, Freq
from czsc.mock import generate_symbol_kines

# 准备数据
df = generate_symbol_kines('000001', '30分钟', '20230101', '20240301')
bars = format_standard_kline(df, freq=Freq.F30)

# 创建 BarGenerator 合成多周期
bg = BarGenerator(base_freq='30分钟', freqs=['30分钟', '日线'], max_count=5000)
for bar in bars:
    bg.update(bar)

# 创建 CzscTrader
ct = CzscTrader(bg, positions=[])
print(f"交易决策器: {ct}")
print(f"品种: {ct.symbol}")
print(f"分析周期: {ct.freqs}")
print(f"最新时间: {ct.end_dt}")
print(f"最新价格: {ct.latest_price}")

交易决策器: <CzscSignals for 000001>
品种: 000001
分析周期: ['30分钟', '日线']
最新时间: 2024-03-01 15:00:00
最新价格: 123.28


In [23]:
# 增量更新 - 模拟实时行情
# 使用 update 方法逐根 K 线更新
df_new = generate_symbol_kines('000001', '30分钟', '20240301', '20240315')
new_bars = format_standard_kline(df_new, freq=Freq.F30)

for bar in new_bars[:5]:  # 只更新前 5 根作为演示
    ct.update(bar)
    print(f"更新: {bar.dt.strftime('%Y-%m-%d %H:%M')}, 收盘={bar.close:.2f}, "
          f"仓位变化={ct.pos_changed}")

print(f"\n最终状态: 时间={ct.end_dt}, 价格={ct.latest_price}")

更新: 2024-03-01 09:30, 收盘=99.26, 仓位变化=False
更新: 2024-03-01 10:00, 收盘=99.22, 仓位变化=False
更新: 2024-03-01 10:30, 收盘=98.89, 仓位变化=False
更新: 2024-03-01 11:00, 收盘=98.25, 仓位变化=False
更新: 2024-03-01 11:30, 收盘=98.49, 仓位变化=False

最终状态: 时间=2024-03-01 11:30:00, 价格=98.49


In [ ]:
# CzscTrader 支持多仓位策略集成
# get_ensemble_pos() 方法可获取多策略集成后的综合仓位
# 支持的集成方法：mean（均值）、vote（投票）、max（最大值）等

# ct.get_ensemble_pos(method='mean')  # 多策略均值
# ct.get_ensemble_pos(method='vote')  # 多策略投票
print("多策略集成方法: mean, vote, max, 或自定义 Callable")

## 8. 策略回测（WeightBacktest）

`WeightBacktest` 是 czsc 的策略回测引擎，支持时序策略（ts）和截面策略（cs）的权重回测。

In [ ]:
from czsc import WeightBacktest
import pandas as pd
import numpy as np

# 构造回测输入数据
# 需要的列：dt（日期时间）、symbol（品种代码）、weight（权重，-1到1）、price（价格）
np.random.seed(42)
dates = pd.date_range('2020-01-01', '2023-12-31', freq='D')
symbols = ['AAPL', 'MSFT', 'GOOGL']

data = []
for dt in dates:
    for sym in symbols:
        data.append({
            'dt': dt,
            'symbol': sym,
            'weight': np.random.uniform(-1, 1),  # 持仓权重，-1为做空，1为做多
            'price': 100 + np.random.randn() * 10,  # 价格
        })

dfw = pd.DataFrame(data)
print(f"回测数据形状: {dfw.shape}")
print(f"品种数量: {dfw['symbol'].nunique()}")
dfw.head()

In [ ]:
# 执行回测
wb = WeightBacktest(dfw, fee_rate=0.0002, weight_type='ts', yearly_days=252)

# 查看回测统计
stats = wb.stats
print("=== 回测统计 ===")
for k, v in stats.items():
    print(f"  {k}: {v}")

In [ ]:
# 查看多空分离统计
print("=== 多头统计 ===")
for k, v in wb.long_stats.items():
    print(f"  {k}: {v}")

print("\n=== 空头统计 ===")
for k, v in wb.short_stats.items():
    print(f"  {k}: {v}")

In [ ]:
# 查看日收益数据
daily_ret = wb.daily_return
print(f"日收益 DataFrame 形状: {daily_ret.shape}")
print(f"日收益列: {daily_ret.columns.tolist()}")
daily_ret.head()

In [ ]:
# 按品种查看交易明细
symbol_daily = wb.get_symbol_daily('AAPL')
print(f"AAPL 日收益数据形状: {symbol_daily.shape}")
symbol_daily.head()

In [ ]:
# 按品种查看交易对
pairs = wb.get_symbol_pairs('AAPL')
print(f"AAPL 交易对数量: {len(pairs)}")
if len(pairs) > 0:
    print(pairs.head())

## 9. 绩效分析与可视化

czsc 提供丰富的绩效分析和可视化工具。

In [ ]:
# daily_performance - 日收益绩效分析
from czsc import daily_performance
import numpy as np

np.random.seed(42)
returns = np.random.normal(0.001, 0.02, 500)

perf = daily_performance(returns)
print("=== 绩效指标 ===")
for k, v in perf.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")

In [ ]:
# top_drawdowns - 最大回撤分析
from czsc import top_drawdowns
import pandas as pd

# 构造净值曲线
np.random.seed(42)
dates = pd.date_range('2020-01-01', periods=500, freq='D')
nav = pd.Series(np.cumprod(1 + np.random.normal(0.001, 0.02, 500)), index=dates)

drawdowns = top_drawdowns(nav, top=5)
print("=== Top 5 回撤 ===")
for i, dd in enumerate(drawdowns):
    print(f"  回撤{i+1}: {dd}")

In [ ]:
# 可视化工具（需要安装 plotly）
# czsc.utils 提供多种回测可视化函数：
#
# - plot_cumulative_returns(): 累计收益曲线
# - plot_drawdown_analysis(): 回撤分析图
# - plot_daily_return_distribution(): 日收益分布
# - plot_monthly_heatmap(): 月度收益热力图
# - plot_backtest_stats(): 综合回测统计（3图组合）
# - plot_colored_table(): 带颜色编码的绩效表格
# - plot_long_short_comparison(): 多空收益对比

from czsc.utils import plot_cumulative_returns
from czsc.mock import generate_daily_returns

# 生成模拟日收益数据
dret = generate_daily_returns(n_strategies=3)
print(f"日收益数据形状: {dret.shape}")
print(f"策略列: {dret.columns.tolist()}")

# 绘制累计收益曲线
fig = plot_cumulative_returns(dret, title="策略累计收益", to_html=False)
fig.show()

In [ ]:
# 综合回测统计图
from czsc.utils import plot_backtest_stats

fig = plot_backtest_stats(dret, ret_col='strategy_A', title='策略A回测统计')
fig.show()

In [ ]:
# 使用 Streamlit 组件库（czsc.svc）进行可视化
# czsc.svc 提供 80+ 个预制组件用于构建量化研究 Streamlit 应用
#
# 主要模块：
# - czsc.svc.backtest: 回测分析组件
# - czsc.svc.factor: 因子分析组件
# - czsc.svc.correlation: 相关性分析组件
# - czsc.svc.statistics: 统计分析组件
#
# 使用示例（在 Streamlit 应用中）：
# import streamlit as st
# from czsc.svc import show_weight_backtest
# show_weight_backtest(dfw)

print("czsc.svc 组件库已就绪，需在 Streamlit 环境中使用")

## 10. 数据源连接器

czsc 支持多种数据源，统一接口封装，位于 `czsc.connectors` 模块。

In [ ]:
# 支持的数据源
print("=== 支持的数据源 ===")
print("  tq_connector: 天勤数据源（期货）")
print("  ts_connector: Tushare 数据源（A股）")
print("  jq_connector: 聚宽数据源（A股）")
print("  ccxt_connector: CCXT 数据源（数字货币）")
print("  research: 研究数据接口")
print("  cooperation: 合作数据接口")

In [ ]:
# DataClient - 统一数据客户端
from czsc import DataClient

# DataClient 提供统一的数据访问接口
# 初始化时需要配置数据源连接信息
# dc = DataClient(connector='tushare', token='your_token')

print("DataClient 支持多种数据源的统一访问")

In [ ]:
# 磁盘缓存管理
from czsc import home_path, get_dir_size, empty_cache_path

print(f"缓存目录: {home_path}")
cache_size = get_dir_size(home_path)
print(f"缓存大小: {cache_size / (1024**2):.1f} MB")

# 清空缓存（谨慎使用）
# empty_cache_path()

## 附录：EDA 探索性分析工具

czsc 提供丰富的探索性分析（EDA）工具函数。

In [ ]:
# EDA 工具函数概览
from czsc import (
    vwap, twap,                           # 加权平均价
    cross_sectional_strategy,             # 截面多空组合构建
    judge_factor_direction,               # 因子方向判断
    monotonicity,                         # 单调性分析
    rolling_layers,                       # 滚动分层
    cal_symbols_factor,                   # 多品种因子计算
    cal_trade_price,                      # 交易价格计算
    make_price_features,                  # 价格特征工程
    turnover_rate,                        # 换手率
    remove_beta_effects,                  # 去除 beta 影响
    weights_simple_ensemble,              # 权重简单集成
    cal_yearly_days,                      # 年度交易日
)

print("EDA 工具函数导入成功！")

In [ ]:
# vwap / twap 示例
import numpy as np

prices = np.array([100.0, 101.0, 102.0, 101.5, 103.0])
volumes = np.array([1000, 1500, 2000, 1200, 1800])

print(f"VWAP（成交量加权平均价）: {vwap(prices, volumes):.2f}")
print(f"TWAP（时间加权平均价）: {twap(prices):.2f}")

In [ ]:
# monotonicity - 单调性分析
seq = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
print(f"递增序列的单调性: {monotonicity(seq):.4f}")

seq2 = [10, 9, 8, 7, 6, 5, 4, 3, 2, 1]
print(f"递减序列的单调性: {monotonicity(seq2):.4f}")

seq3 = [1, 3, 2, 4, 3, 5, 4, 6, 5, 7]
print(f"波动序列的单调性: {monotonicity(seq3):.4f}")

## 附录：策略开发框架

czsc 提供策略抽象基类，支持结构化的策略定义。

In [ ]:
# CzscStrategyBase - 策略开发基类
from czsc import CzscStrategyBase
from abc import abstractmethod

# 策略开发模式：
# 1. 继承 CzscStrategyBase
# 2. 实现 positions 属性（返回 Position 列表）
# 3. 框架自动推导：signals_config、freqs、base_freq
#
# class MyStrategy(CzscStrategyBase):
#     @property
#     def positions(self):
#         return [pos1, pos2, ...]  # 定义持仓策略
#
# # 回测
# strategy = MyStrategy(symbol='000001')
# result = strategy.backtest(bars)

# CzscJsonStrategy - 从 JSON 文件加载仓位定义
from czsc import CzscJsonStrategy

# strategy = CzscJsonStrategy(symbol='000001', files_position=['pos_A.json'])
# result = strategy.backtest(bars)

print("策略开发框架说明完成")
print("  CzscStrategyBase: 继承式策略开发")
print("  CzscJsonStrategy: JSON配置式策略开发")
print("  支持: backtest() 回测, replay() 回放, save_positions() 保存策略")

In [ ]:
# 策略回放与研究工具
from czsc.core import run_research, run_replay, run_optimize_batch

# run_research(): 执行策略研究
# run_replay(): 执行策略回放
# run_optimize_batch(): 批量参数优化

# 参数优化工具
from czsc.core import OpensOptimize, ExitsOptimize
from czsc.core import build_open_optim_positions, build_exit_optim_positions

print("策略研究工具导入成功")
print("  run_research: 策略研究")
print("  run_replay: 策略回放")
print("  run_optimize_batch: 批量参数优化")
print("  OpensOptimize: 开仓参数优化")
print("  ExitsOptimize: 平仓参数优化")

In [ ]:
# 常用工具函数
from czsc import (
    x_round,           # 精确截断小数
    freqs_sorted,      # K线周期排序
    create_grid_params, # 创建网格参数
    save_json,         # 保存 JSON
    read_json,         # 读取 JSON
    dill_dump,         # 序列化对象
    dill_load,         # 反序列化对象
    DiskCache,         # 磁盘缓存
)

# x_round - 精确截断（非四舍五入）
print(f"x_round(3.14159, 2) = {x_round(3.14159, 2)}")
print(f"x_round(3.14999, 2) = {x_round(3.14999, 2)}")

# freqs_sorted - K线周期排序
freqs = ['日线', '30分钟', '5分钟', '60分钟', '1分钟']
print(f"排序前: {freqs}")
print(f"排序后: {freqs_sorted(freqs)}")

# create_grid_params - 创建网格搜索参数
grid = create_grid_params(prefix='model', n1=[5, 10, 20], n2=[3, 5])
print(f"网格参数数量: {len(grid['model'])}")